In [41]:
#### cach 1 cũ
import os
import cv2
import numpy as np
from tqdm import tqdm

input_dir = r"C:\Users\Admin\Documents\xu_ly_anh_khop\input"
output_dir = r"C:\Users\Admin\Documents\xu_ly_anh_khop\output_qroi8_2"
os.makedirs(output_dir, exist_ok=True)

TARGET_SIZE = 8
STANDARD_WIDTH = 64

# giam nhieu anh
def denoise(img):
    return cv2.GaussianBlur(img, (3,3), 0)

# quay góc ảnh cho đồng nhất
def auto_rotate_knee(gray):
    edges = cv2.Canny(gray, 40, 120)
    lines = cv2.HoughLines(edges, 1, np.pi/180, 120)
    if lines is None:
        return gray

    angles = [(theta - np.pi/2)*180/np.pi for rho,theta in lines[:,0]]
    median_angle = np.median(angles)

    h,w = gray.shape
    M = cv2.getRotationMatrix2D((w//2,h//2), median_angle, 1.0)
    return cv2.warpAffine(gray, M, (w,h),
                          flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_REPLICATE)

# xác định vị trí khe khớp 
def find_joint_band(gray):
    edges = cv2.Canny(gray,30,120)
    proj = np.sum(edges, axis=1)
    y = np.argmax(proj)

    band = int(gray.shape[0]*0.12)
    return max(0,y-band), min(gray.shape[0],y+band)

# lấy vùng xương
def crop_bone_width(gray_band):
    _, th = cv2.threshold(gray_band, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    contours,_ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return gray_band

    c = max(contours, key=cv2.contourArea)
    x,y,w,h = cv2.boundingRect(c)
    return gray_band[:, x:x+w], w

# scale theo khe
def normalize_scale(roi, width):
    scale = STANDARD_WIDTH / width
    h = int(roi.shape[0]*scale)
    return cv2.resize(roi, (STANDARD_WIDTH, h), interpolation=cv2.INTER_LINEAR)

# chuan hóa ánh sáng
def normalize_clahe(img):
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    return clahe.apply(img)


# lấy tâm
def center_crop_square(img):
    h,w = img.shape
    size = min(h,w)
    return img[h//2-size//2:h//2+size//2,
               w//2-size//2:w//2+size//2]


# main
for root,_,files in os.walk(input_dir):
    for file in tqdm(files):
        if not file.lower().endswith((".png",".jpg",".jpeg")):
            continue

        path = os.path.join(root,file)
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue

        img = denoise(img)
       
        img = auto_rotate_knee(img)

        y1,y2 = find_joint_band(img)
        band = img[y1:y2,:]
         
        img = normalize_clahe(img)
        
        roi, w = crop_bone_width(band)
        roi = normalize_scale(roi, w)

        roi = center_crop_square(roi)

        roi_small = cv2.resize(roi,(TARGET_SIZE,TARGET_SIZE),
                               interpolation=cv2.INTER_AREA)

        # giữ nguyên cấu trúc thư mục
        rel_path = os.path.relpath(path, input_dir)

        save_path = os.path.join(output_dir, rel_path)

        # tạo folder nếu chưa có
        os.makedirs(os.path.dirname(save_path), exist_ok=True)

        cv2.imwrite(save_path, roi_small)

print("Done pipeline 1")

0it [00:00, ?it/s]
0it [00:00, ?it/s]
100%|██████████| 206/206 [00:00<00:00, 409.39it/s]
0it [00:00, ?it/s]
100%|██████████| 206/206 [00:00<00:00, 425.15it/s]

Done pipeline 1


In [47]:
import os
import cv2
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

DATA_DIR = r"C:\Users\Admin\Documents\xu_ly_anh_khop\output_qroi8_2"

def load_dataset(data_dir):
    X, y = [], []
    label_map = {
        "0Normal":0,
        "1Doubtful":1,
        "2Mild":2,
        "3Moderate":3,
        "4Severe":4
    }

    for root, dirs, files in os.walk(data_dir):
        for file in files:
            if not file.lower().endswith((".png",".jpg",".jpeg")):
                continue

            path = os.path.join(root, file)
            label_name = os.path.basename(os.path.dirname(path))
            if label_name not in label_map:
                continue

            img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

            # chuẩn hóa + scale cho quantum (quan trọng)
            img = (img / 255.0) * np.pi

            X.append(img.flatten())  # 64 chiều
            y.append(label_map[label_name])

    return np.array(X), np.array(y)

X, y = load_dataset(DATA_DIR)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# mo hinh quantum
n_qubits = 8
n_layers = 8   # train ổn định hơn
dev = qml.device("default.qubit", wires=n_qubits)

def encode_layer(features):
    for i in range(n_qubits):
        qml.Hadamard(wires=i) # 
        qml.RY(features[i], wires=i)

def variational_layer(weights):
    for i in range(n_qubits):
        qml.RY(weights[i], wires=i)
        qml.RZ(weights[i], wires=i) # 

    for i in range(n_qubits - 1):
        qml.CNOT(wires=[i, i+1])

@qml.qnode(dev)
def quantum_circuit(x, weights):

    chunks = np.array_split(x, n_layers)

    for i in range(n_layers):
        chunk = chunks[i]

        # nếu thiếu thì pad thêm
        if len(chunk) < n_qubits:
            chunk = np.pad(chunk, (0, n_qubits - len(chunk)))

        encode_layer(chunk[:n_qubits])
        variational_layer(weights[i])

    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]


# CLASSICAL PART

n_classes = 5

weights = pnp.random.randn(n_layers, n_qubits, requires_grad=True)
W_classical = pnp.random.randn(n_qubits, n_classes, requires_grad=True)

def softmax(x):
    e = pnp.exp(x - pnp.max(x))  # tránh overflow
    return e / pnp.sum(e)

def forward(x, weights, W_classical):
    q_out = pnp.array(quantum_circuit(x, weights))
    logits = pnp.dot(q_out, W_classical)
    return softmax(logits)

def loss_fn(weights, W_classical, X, y):
    loss = 0
    for i in range(len(X)):
        probs = forward(X[i], weights, W_classical)
        loss -= pnp.log(probs[y[i]] + 1e-9)
    return loss / len(X)

n_classes = 5

weights = pnp.random.randn(n_layers, n_qubits, requires_grad=True)
W_classical = pnp.random.randn(n_qubits, n_classes, requires_grad=True)


In [40]:
opt = qml.AdamOptimizer(0.01)
epochs = 60
batch_size = 32

for epoch in range(epochs):

    # shuffle
    idx = np.random.permutation(len(X_train))

    for i in range(0, len(X_train), batch_size):
        batch_idx = idx[i:i+batch_size]
        X_batch = X_train[batch_idx]
        y_batch = y_train[batch_idx]

        weights, W_classical = opt.step(
            lambda w,c: loss_fn(w,c,X_batch,y_batch),
            weights, W_classical
        )

    # tính loss nhanh trên 100 sample
    train_loss = loss_fn(weights, W_classical, X_train[:100], y_train[:100])
    print(f"Epoch {epoch+1} | Loss: {train_loss:.4f}")


def predict(X):
    preds = []
    for i in range(len(X)):
        probs = forward(X[i], weights, W_classical)
        preds.append(int(np.argmax(probs)))
    return np.array(preds)

y_pred = predict(X_test)
acc = accuracy_score(y_test, y_pred)

print("Test Accuracy:", acc)

Epoch 1 | Loss: 1.3601
Epoch 2 | Loss: 1.3528
Epoch 3 | Loss: 1.3423
Epoch 4 | Loss: 1.3592
Epoch 5 | Loss: 1.3439
Epoch 6 | Loss: 1.3584
Epoch 7 | Loss: 1.3588
Epoch 8 | Loss: 1.3507
Epoch 9 | Loss: 1.3549
Epoch 10 | Loss: 1.3585
Epoch 11 | Loss: 1.3721
Epoch 12 | Loss: 1.3612
Epoch 13 | Loss: 1.3516
Epoch 14 | Loss: 1.3512
Epoch 15 | Loss: 1.3686
Epoch 16 | Loss: 1.3444
Epoch 17 | Loss: 1.3621
Epoch 18 | Loss: 1.3380
Epoch 19 | Loss: 1.3572
Epoch 20 | Loss: 1.3597
Epoch 21 | Loss: 1.3395
Epoch 22 | Loss: 1.3502
Epoch 23 | Loss: 1.3459
Epoch 24 | Loss: 1.3622
Epoch 25 | Loss: 1.3310
Epoch 26 | Loss: 1.3416
Epoch 27 | Loss: 1.3375
Epoch 28 | Loss: 1.3533
Epoch 29 | Loss: 1.3394
Epoch 30 | Loss: 1.3475
Epoch 31 | Loss: 1.3507
Epoch 32 | Loss: 1.3370
Epoch 33 | Loss: 1.3569
Epoch 34 | Loss: 1.3519
Epoch 35 | Loss: 1.3622
Epoch 36 | Loss: 1.3362
Epoch 37 | Loss: 1.3486
Epoch 38 | Loss: 1.3400
Epoch 39 | Loss: 1.3426
Epoch 40 | Loss: 1.3688
Epoch 41 | Loss: 1.3557
Epoch 42 | Loss: 1.3504
E

# cach 2
Epoch 1 | Loss: 1.3724
Epoch 2 | Loss: 1.3590
Epoch 3 | Loss: 1.3552
Epoch 4 | Loss: 1.3602
Epoch 5 | Loss: 1.3581
Epoch 6 | Loss: 1.3768
Epoch 7 | Loss: 1.3546
Epoch 8 | Loss: 1.3686
Epoch 9 | Loss: 1.3709
Epoch 10 | Loss: 1.3679

Test Accuracy: 0.396969696969697

Epoch 58 | Loss: 1.3489
Epoch 59 | Loss: 1.3586
Epoch 60 | Loss: 1.3857
Test Accuracy: 0.4015151515151515

# cach 1
Epoch 1 | Loss: 1.4469
Epoch 2 | Loss: 1.4433
Epoch 3 | Loss: 1.4380
Epoch 4 | Loss: 1.4481
Epoch 5 | Loss: 1.4439
Epoch 6 | Loss: 1.4526
Epoch 7 | Loss: 1.4532
Epoch 8 | Loss: 1.4496
Epoch 9 | Loss: 1.4640
Epoch 10 | Loss: 1.4521

Test Accuracy: 0.4106060606060606